In [1]:
import numpy as np


class DecisionTreeRegressorNode:

  def __init__(
      self,
      feature=None,
      threshold=None,
      left=None,
      right=None,
      *,
      value=None,
  ):
    self.feature = feature
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value

  def is_leaf(self):
    return self.value is not None


class DecisionTreeRegressorScratch:

  def __init__(self, max_depth=3, min_samples_split=2):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.root = None

  def _best_split(self, X, y):
    best_variance_reduction = -1
    best_feat, best_thresh = None, None
    parent_variance = np.var(y) * len(y)

    n_samples, n_features = X.shape

    for feat_idx in range(n_features):
      thresholds = np.unique(X[:, feat_idx])
      for thresh in thresholds:
        left_mask = X[:, feat_idx] <= thresh
        right_mask = ~left_mask

        if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
          continue

        y_left, y_right = y[left_mask], y[right_mask]
        left_variance = np.var(y_left) * len(y_left)
        right_variance = np.var(y_right) * len(y_right)

        reduction = parent_variance - (left_variance + right_variance)

        if reduction > best_variance_reduction:
          best_variance_reduction = reduction
          best_feat = feat_idx
          best_thresh = thresh

    return best_feat, best_thresh

  def _build_tree(self, X, y, depth=0):
    n_samples = X.shape[0]

    if depth >= self.max_depth or n_samples < self.min_samples_split:
      return DecisionTreeRegressorNode(value=np.mean(y))

    feat_idx, thresh = self._best_split(X, y)
    if feat_idx is None:
      return DecisionTreeRegressorNode(value=np.mean(y))

    left_mask = X[:, feat_idx] <= thresh
    left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
    right_child = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)

    return DecisionTreeRegressorNode(
        feature=feat_idx,
        threshold=thresh,
        left=left_child,
        right=right_child,
    )

  def fit(self, X, y):
    self.root = self._build_tree(X, y)

  def _predict_row(self, x, node):
    if node.is_leaf():
      return node.value
    if x[node.feature] <= node.threshold:
      return self._predict_row(x, node.left)
    return self._predict_row(x, node.right)

  def predict(self, X):
    return np.array([self._predict_row(x, self.root) for x in X])


class GradientBoostingRegressorScratch:

  def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
    self.n_estimators = n_estimators
    self.learning_rate = learning_rate
    self.max_depth = max_depth
    self.base_pred = None
    self.trees = []

  def fit(self, X, y):
    # 1. Initialize ensemble with optimal constant base prediction: F_0(x) = mean(y)
    self.base_pred = np.mean(y)
    F_m = np.full(y.shape, self.base_pred)

    self.trees = []

    for _ in range(self.n_estimators):
      # 2. Compute negative gradient / pseudo-residuals: r_im = y_i - F_{m-1}(x_i)
      residuals = y - F_m

      # 3. Fit a shallow regression tree h_m(x) to the pseudo-residuals
      tree = DecisionTreeRegressorScratch(max_depth=self.max_depth)
      tree.fit(X, residuals)

      # 4. Update model in function space: F_m(x) = F_{m-1}(x) + learning_rate * h_m(x)
      h_m = tree.predict(X)
      F_m += self.learning_rate * h_m

      self.trees.append(tree)

  def predict(self, X):
    # F(x) = F_0 + eta * sum(h_m(x))
    preds = np.full(X.shape[0], self.base_pred)
    for tree in self.trees:
      preds += self.learning_rate * tree.predict(X)
    return preds


# =========================================================================
# DEMO EXECUTION
# =========================================================================
if __name__ == "__main__":
  np.random.seed(42)

  # Synthetic non-linear continuous regression target: y = sin(x) + noise
  X_train = np.linspace(-3, 3, 100).reshape(-1, 1)
  y_train = np.sin(X_train).ravel() + np.random.normal(0, 0.1, X_train.shape[0])

  # Train Gradient Boosting Regressor from scratch
  gbm = GradientBoostingRegressorScratch(
      n_estimators=50, learning_rate=0.1, max_depth=2
  )
  gbm.fit(X_train, y_train)

  # Generate predictions
  y_pred = gbm.predict(X_train)
  mse = np.mean((y_train - y_pred) ** 2)

  print("=" * 60)
  print("  GRADIENT BOOSTING REGRESSOR FROM SCRATCH (NUMPY)")
  print("=" * 60)
  print(f"Number of Boosting Stages (Trees): {gbm.n_estimators}")
  print(f"Shrinkage / Learning Rate (eta)  : {gbm.learning_rate}")
  print(f"Base Initial Value (F0)          : {gbm.base_pred:.4f}")
  print(f"Final Training Mean Squared Error: {mse:.6f}")
  print("=" * 60)

  GRADIENT BOOSTING REGRESSOR FROM SCRATCH (NUMPY)
Number of Boosting Stages (Trees): 50
Shrinkage / Learning Rate (eta)  : 0.1
Base Initial Value (F0)          : -0.0104
Final Training Mean Squared Error: 0.004540
